# Generate Frozen YOLOWorld Models — COCO-80 Vocabulary

Generates one frozen `.pt` file per Places365 scene using **`yolov8s-worldv2.pt`** (base model, no fine-tuning required).

Each frozen model has its CLIP text embeddings baked in — loadable on any device **without** calling `set_classes()` at runtime.

### What this notebook does
1. Installs ultralytics and downloads `yolov8s-worldv2.pt`
2. Loads the 365-scene → COCO-80 class mapping (embedded, no upload needed)
3. For each scene: `model.set_classes(classes)` → `model.save(scene.pt)`
4. Benchmarks frozen load vs dynamic `set_classes()` latency
5. Downloads a zip of all generated models

> **Runtime:** Use GPU (T4 / A100) for fastest `set_classes()` (CLIP runs on GPU).  
> Estimated time: ~2–5 min for all 365 scenes on T4.

---

## 1 — Install dependencies

In [ ]:
!pip install -q ultralytics
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2 — Configuration

In [ ]:
from pathlib import Path

# Base YOLOWorld model (auto-downloaded by ultralytics on first use)
MODEL_NAME = 'yolov8s-worldv2.pt'

# Output directory for frozen per-scene models
OUT_DIR = Path('scene_models_coco80')
OUT_DIR.mkdir(exist_ok=True)

# Optional: generate only a subset of scenes (None = all 365)
SCENE_FILTER = None
# Example: SCENE_FILTER = ['living_room', 'kitchen', 'office', 'street', 'park']

# Skip scenes whose .pt file already exists (safe to re-run)
SKIP_EXISTING = True

# Google Drive — set True to persist output across Colab sessions
USE_DRIVE = False
DRIVE_DIR = Path('/content/drive/MyDrive/scene_models_coco80')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = DRIVE_DIR
    OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output dir : {OUT_DIR.resolve()}')
print(f'Base model : {MODEL_NAME}')

## 3 — Scene → COCO-80 Class Mapping

365 Places365 scenes mapped to COCO-80 classes.  
This is the same mapping used by `backend/data/context.db` — no file upload needed.

In [ ]:
import json, statistics

# ── Embedded COCO-80 scene→classes mapping (from context.db, 365 scenes) ──
_SCENES_JSON = '{"airfield":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"airplane_cabin":["backpack","bed","bench","book","cell phone","chair","clock","cup","handbag","person","suitcase","umbrella"],"airport_terminal":["airplane","backpack","bench","cell phone","handbag","person","suitcase","tv","umbrella"],"alcove":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"alley":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"amphitheater":["bench","book","clock","person","potted plant","vase"],"amusement_arcade":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"amusement_park":["bench","bicycle","bird","dog","frisbee","kite","person","potted plant","skateboard","sports ball"],"apartment_building_outdoor":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"aquarium":["backpack","book","chair","clock","person","potted plant","vase"],"aqueduct":["bench","book","clock","fire hydrant","person","potted plant","traffic light","vase"],"arcade":["backpack","book","chair","clock","person","potted plant","vase"],"arch":["bench","book","clock","person","potted plant","vase"],"archaelogical_excavation":["bench","book","clock","person","potted plant","vase"],"archive":["backpack","book","chair","clock","person","potted plant","vase"],"arena_hockey":["backpack","baseball bat","baseball glove","bench","chair","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"arena_performance":["backpack","baseball bat","baseball glove","bench","chair","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"arena_rodeo":["backpack","baseball bat","baseball glove","bench","chair","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"army_base":["bench","book","clock","person","potted plant","vase"],"art_gallery":["backpack","bench","book","chair","clock","person","potted plant","vase"],"art_school":["backpack","book","chair","clock","person","potted plant","vase"],"art_studio":["backpack","book","cell phone","chair","clock","keyboard","laptop","mouse","person","potted plant","scissors","tv","vase"],"artists_loft":["backpack","book","chair","clock","person","potted plant","vase"],"assembly_line":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"athletic_field_outdoor":["bench","bicycle","bird","cow","dog","frisbee","horse","kite","person","sheep","skateboard","sports ball"],"atrium_public":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"attic":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","suitcase","teddy bear","tv","vase"],"auditorium":["backpack","book","chair","clock","person","potted plant","vase"],"auto_factory":["book","car","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","truck","tv"],"auto_showroom":["bottle","bowl","car","chair","cup","dining table","fork","knife","person","spoon","truck","wine glass"],"badlands":["backpack","person"],"bakery_shop":["backpack","bottle","bowl","cake","cell phone","chair","cup","dining table","donut","fork","handbag","knife","person","spoon","umbrella","wine glass"],"balcony_exterior":["backpack","bench","bicycle","bird","bottle","bus","car","cat","cell phone","chair","cow","dog","handbag","horse","motorcycle","person","potted plant","sheep","traffic light","truck","umbrella"],"balcony_interior":["backpack","bench","bicycle","bird","bottle","bus","car","cat","cell phone","chair","cow","dog","handbag","horse","motorcycle","person","potted plant","sheep","traffic light","truck","umbrella"],"ball_pit":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"ballroom":["baseball bat","baseball glove","bench","chair","clock","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket","vase"],"bamboo_forest":["backpack","bird","cat","dog","person"],"bank_vault":["book","bottle","bowl","cell phone","chair","clock","cup","dining table","fork","keyboard","knife","laptop","mouse","person","scissors","spoon","tv","wine glass"],"banquet_hall":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","vase","wine glass"],"bar":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","tv","wine glass"],"barn":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light"],"barndoor":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"baseball_field":["baseball bat","baseball glove","bench","bicycle","bird","cow","dog","frisbee","horse","kite","person","sheep","skateboard","sports ball"],"basement":["bed","bicycle","book","bottle","chair","clock","couch","cup","person","potted plant","remote","suitcase","tv","vase"],"basketball_court_indoor":["backpack","baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"bathroom":["bed","book","bottle","chair","clock","couch","cup","hair drier","person","potted plant","remote","sink","toilet","toothbrush","tv","vase"],"bazaar_indoor":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"bazaar_outdoor":["backpack","bench","bicycle","book","bottle","bus","car","cell phone","clock","handbag","motorcycle","person","potted plant","traffic light","truck","umbrella","vase"],"beach":["bird","boat","dog","frisbee","kite","person","surfboard","umbrella"],"beach_house":["bench","bicycle","bird","boat","cat","cow","dog","frisbee","horse","kite","person","potted plant","sheep","surfboard","umbrella"],"beauty_salon":["backpack","book","bottle","bowl","chair","clock","cup","dining table","fork","knife","person","potted plant","spoon","vase","wine glass"],"bedchamber":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"bedroom":["bed","book","bottle","cell phone","chair","clock","couch","cup","laptop","person","potted plant","remote","teddy bear","tv","vase"],"beer_garden":["bench","bicycle","bird","cat","dog","frisbee","kite","person","potted plant","skateboard","sports ball","vase"],"beer_hall":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"berth":["backpack","bench","cell phone","handbag","person","suitcase","umbrella"],"biology_laboratory":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"boardwalk":["backpack","bench","bicycle","bird","boat","cat","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","sports ball","surfboard","traffic light","umbrella"],"boat_deck":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"boathouse":["airplane","bench","bicycle","boat","bus","car","clock","fire hydrant","motorcycle","parking meter","person","potted plant","stop sign","surfboard","traffic light","train","truck","umbrella"],"bookstore":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"booth_indoor":["backpack","book","cell phone","chair","clock","keyboard","laptop","mouse","person","potted plant","scissors","tv","vase"],"botanical_garden":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light","vase"],"bow_window_indoor":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"bowling_alley":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"boxing_ring":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"bridge":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"building_facade":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"bullring":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"burial_chamber":["backpack","book","chair","clock","person","potted plant","vase"],"bus_interior":["backpack","bench","bus","cell phone","handbag","person","suitcase","umbrella"],"bus_station_indoor":["airplane","backpack","bench","bicycle","boat","bus","car","cell phone","handbag","motorcycle","parking meter","person","stop sign","suitcase","traffic light","train","truck","umbrella"],"butchers_shop":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","umbrella","wine glass"],"butte":["backpack","person"],"cabin_outdoor":["bed","bench","bicycle","bird","book","cat","chair","clock","cow","cup","dog","horse","person","potted plant","sheep"],"cafeteria":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"campsite":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"campus":["bench","book","clock","person","potted plant","vase"],"canal_natural":["boat","person","surfboard","umbrella"],"canal_urban":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"candy_store":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","wine glass"],"canyon":["backpack","person"],"car_interior":["backpack","bench","car","cell phone","handbag","person","suitcase","umbrella"],"carrousel":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"castle":["bench","book","clock","person","potted plant","vase"],"catacomb":["backpack","book","chair","clock","person","potted plant","vase"],"cemetery":["bench","book","clock","person","potted plant","vase"],"chalet":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"chemistry_lab":["book","bottle","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"childs_room":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"church_indoor":["backpack","bench","book","chair","clock","person","potted plant","vase"],"church_outdoor":["bench","book","clock","person","potted plant","vase"],"classroom":["backpack","book","chair","clock","laptop","person","potted plant","tv","vase"],"clean_room":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"cliff":["backpack","person"],"closet":["backpack","bed","book","bottle","chair","clock","couch","cup","handbag","person","potted plant","remote","tie","tv","umbrella","vase"],"clothing_store":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","wine glass"],"coast":["boat","person","surfboard","umbrella"],"cockpit":["airplane","backpack","bench","cell phone","handbag","person","suitcase","umbrella"],"coffee_shop":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","umbrella","wine glass"],"computer_room":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"conference_center":["backpack","book","bottle","cell phone","chair","clock","cup","laptop","person","potted plant","tv","vase"],"conference_room":["book","bottle","cell phone","chair","clock","cup","keyboard","laptop","mouse","person","scissors","tv"],"construction_site":["backpack","car","person","truck"],"corn_field":["backpack","bird","cat","cow","dog","horse","person","sheep"],"corral":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light"],"corridor":["book","cell phone","chair","clock","fire hydrant","keyboard","laptop","mouse","person","potted plant","scissors","tv"],"cottage":["bed","bench","bicycle","bird","cat","chair","cow","dog","horse","person","potted plant","sheep"],"courthouse":["bench","book","clock","person","potted plant","vase"],"courtyard":["backpack","bench","bicycle","bird","book","bottle","bus","car","cat","cell phone","clock","cow","dog","handbag","horse","motorcycle","person","potted plant","sheep","traffic light","truck","umbrella","vase"],"creek":["boat","person","surfboard","umbrella"],"crevasse":["boat","person","surfboard","umbrella"],"crosswalk":["airplane","backpack","bench","bicycle","boat","bottle","bus","car","cell phone","handbag","motorcycle","parking meter","person","stop sign","traffic light","train","truck","umbrella"],"dam":["backpack","bench","boat","car","clock","fire hydrant","person","potted plant","surfboard","traffic light","truck","umbrella"],"delicatessen":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"department_store":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","wine glass"],"desert_road":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"desert_sand":["backpack","person"],"desert_vegetation":["backpack","person"],"diner_outdoor":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"dining_hall":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"dining_room":["bed","book","bottle","bowl","chair","clock","couch","cup","dining table","fork","knife","person","potted plant","remote","spoon","tv","vase","wine glass"],"discotheque":["baseball bat","baseball glove","bench","bottle","bowl","chair","cup","dining table","fork","frisbee","kite","knife","person","skateboard","skis","snowboard","spoon","sports ball","surfboard","tennis racket","wine glass"],"doorway_outdoor":["backpack","bench","bicycle","bird","bottle","bus","car","cat","cell phone","cow","dog","handbag","horse","motorcycle","person","potted plant","sheep","traffic light","truck","umbrella"],"dorm_room":["backpack","bed","book","bottle","cell phone","chair","clock","couch","cup","laptop","person","potted plant","remote","tv","vase"],"downtown":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"dressing_room":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"driveway":["bench","bicycle","bird","car","cat","cow","dog","horse","person","potted plant","sheep","truck"],"drugstore":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"elevator_door":["backpack","book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"elevator_lobby":["backpack","bench","cell phone","chair","clock","couch","handbag","person","potted plant","suitcase","umbrella","vase"],"elevator_shaft":["backpack","book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"embassy":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"engine_room":["backpack","bench","book","cell phone","chair","clock","handbag","keyboard","laptop","mouse","person","scissors","suitcase","tv","umbrella"],"entrance_hall":["backpack","bench","cell phone","handbag","person","suitcase","umbrella"],"escalator_indoor":["backpack","bench","book","bottle","bowl","cell phone","chair","clock","cup","dining table","fork","handbag","keyboard","knife","laptop","mouse","person","scissors","spoon","suitcase","tv","umbrella","wine glass"],"excavation":["backpack","car","person","truck"],"fabric_store":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","wine glass"],"farm":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep","truck"],"fastfood_restaurant":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"field_cultivated":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light"],"field_road":["airplane","bicycle","bird","boat","bus","car","cow","dog","horse","motorcycle","parking meter","person","sheep","stop sign","traffic light","train","truck"],"field_wild":["backpack","bird","cat","cow","dog","horse","person","sheep"],"fire_escape":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"fire_station":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"fishpond":["bench","bicycle","bird","boat","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","sports ball","surfboard","traffic light","umbrella"],"flea_market_indoor":["apple","backpack","banana","bottle","bowl","chair","cup","dining table","fork","handbag","knife","orange","person","spoon","umbrella","wine glass"],"florist_shop_indoor":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","umbrella","wine glass"],"food_court":["apple","banana","bottle","bowl","cake","chair","cup","dining table","donut","fork","knife","orange","person","pizza","sandwich","spoon","wine glass"],"football_field":["bench","bicycle","bird","cow","dog","frisbee","horse","kite","person","sheep","skateboard","sports ball"],"forest_broadleaf":["backpack","bird","cat","dog","person"],"forest_path":["backpack","bird","cat","dog","person"],"forest_road":["airplane","backpack","bench","bicycle","bird","boat","bus","car","cat","clock","dog","fire hydrant","frisbee","kite","motorcycle","parking meter","person","potted plant","skateboard","sports ball","stop sign","traffic light","train","truck"],"formal_garden":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light","vase"],"fountain":["bench","book","clock","person","potted plant","vase"],"galley":["backpack","bench","cell phone","handbag","person","suitcase","umbrella"],"garage_indoor":["bed","bicycle","book","bottle","car","chair","clock","couch","cup","motorcycle","person","potted plant","remote","truck","tv","vase"],"garage_outdoor":["airplane","bench","bicycle","bird","boat","bus","car","cat","cow","dog","horse","motorcycle","parking meter","person","potted plant","sheep","stop sign","traffic light","train","truck"],"gas_station":["airplane","backpack","bench","bicycle","boat","bottle","bus","car","cell phone","handbag","motorcycle","parking meter","person","stop sign","traffic light","train","truck","umbrella"],"gazebo_exterior":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"general_store_indoor":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","wine glass"],"general_store_outdoor":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"gift_shop":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","umbrella","wine glass"],"glacier":["boat","person","surfboard","umbrella"],"golf_course":["backpack","bench","bicycle","bird","cat","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","sports ball","traffic light"],"greenhouse_indoor":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"greenhouse_outdoor":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"grotto":["backpack","boat","person","surfboard","umbrella"],"gymnasium_indoor":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"hangar_indoor":["airplane","backpack","bench","cell phone","handbag","person","suitcase","truck","umbrella"],"hangar_outdoor":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"harbor":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"hardware_store":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","wine glass"],"hayfield":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light"],"heliport":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"highway":["airplane","bench","bicycle","boat","bus","car","clock","fire hydrant","motorcycle","parking meter","person","potted plant","stop sign","traffic light","train","truck"],"home_office":["bed","book","bottle","cell phone","chair","clock","couch","cup","keyboard","laptop","mouse","person","potted plant","remote","scissors","tv","vase"],"home_theater":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"hospital":["backpack","bed","bench","bicycle","bottle","bus","car","cell phone","chair","clock","handbag","motorcycle","person","traffic light","truck","tv","umbrella"],"hospital_room":["bed","book","bottle","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"hot_spring":["boat","person","surfboard","umbrella"],"hotel_outdoor":["backpack","bed","bench","bicycle","bottle","bus","car","cell phone","chair","clock","couch","handbag","motorcycle","person","potted plant","remote","suitcase","traffic light","truck","tv","umbrella","vase"],"hotel_room":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","suitcase","tv","vase"],"house":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"hunting_lodge_outdoor":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"ice_cream_parlor":["bottle","bowl","chair","cup","dining table","fork","knife","person","skis","spoon","wine glass"],"ice_floe":["boat","person","skis","surfboard","umbrella"],"ice_shelf":["boat","person","skis","surfboard","umbrella"],"ice_skating_rink_indoor":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"ice_skating_rink_outdoor":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","skis","sports ball"],"iceberg":["boat","person","surfboard","umbrella"],"igloo":["bench","bicycle","bird","boat","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","sports ball","surfboard","traffic light","umbrella"],"industrial_area":["backpack","car","person","truck"],"inn_outdoor":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"islet":["boat","person","surfboard","umbrella"],"jacuzzi_indoor":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"jail_cell":["backpack","book","chair","clock","person","potted plant","vase"],"japanese_garden":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep","vase"],"jewelry_shop":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","umbrella","wine glass"],"junkyard":["backpack","car","person","truck"],"kasbah":["bench","book","clock","person","potted plant","vase"],"kennel_outdoor":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"kindergarden_classroom":["backpack","book","chair","clock","laptop","person","potted plant","tv","vase"],"kitchen":["bed","book","bottle","bowl","chair","clock","couch","cup","dining table","fork","knife","microwave","oven","person","potted plant","refrigerator","remote","sink","spoon","toaster","tv","vase"],"lagoon":["boat","person","surfboard","umbrella"],"lake_natural":["bird","boat","person","surfboard","umbrella"],"landfill":["backpack","car","person","truck"],"landing_deck":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"laundromat":["bed","book","bottle","cell phone","chair","clock","couch","cup","keyboard","laptop","mouse","person","potted plant","remote","scissors","tv","vase"],"lawn":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"lecture_room":["backpack","book","chair","clock","laptop","person","potted plant","vase"],"legislative_chamber":["backpack","book","chair","clock","person","potted plant","vase"],"library_indoor":["backpack","book","chair","clock","laptop","person","potted plant","vase"],"library_outdoor":["backpack","bench","book","chair","clock","laptop","person","potted plant","vase"],"lighthouse":["bench","book","clock","person","potted plant","vase"],"living_room":["bed","book","bottle","cat","chair","clock","couch","cup","dog","person","potted plant","remote","tv","vase"],"loading_dock":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"lobby":["book","cell phone","chair","clock","couch","keyboard","laptop","mouse","person","potted plant","scissors","suitcase","tv","vase"],"lock_chamber":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"locker_room":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"mansion":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"manufactured_home":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"market_indoor":["apple","backpack","banana","bottle","bowl","chair","cup","dining table","fork","handbag","knife","orange","person","spoon","umbrella","wine glass"],"market_outdoor":["apple","backpack","banana","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","orange","person","traffic light","truck","umbrella"],"marsh":["backpack","bird","boat","cat","dog","person","surfboard","umbrella"],"martial_arts_gym":["backpack","baseball bat","baseball glove","bench","bottle","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"mausoleum":["bench","book","clock","person","potted plant","vase"],"medina":["backpack","bench","bicycle","book","bottle","bus","car","cell phone","clock","handbag","motorcycle","person","potted plant","traffic light","truck","umbrella","vase"],"mezzanine":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"moat_water":["bench","book","clock","person","potted plant","vase"],"mosque_outdoor":["bench","book","clock","person","potted plant","vase"],"motel":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"mountain":["backpack","person"],"mountain_path":["airplane","backpack","bench","bicycle","boat","bus","car","clock","fire hydrant","motorcycle","parking meter","person","potted plant","stop sign","traffic light","train","truck"],"mountain_snowy":["backpack","boat","person","surfboard","umbrella"],"movie_theater_indoor":["backpack","book","chair","clock","person","potted plant","vase"],"museum_indoor":["backpack","bench","book","chair","clock","person","potted plant","vase"],"museum_outdoor":["bench","book","clock","person","potted plant","vase"],"music_studio":["backpack","book","cell phone","chair","clock","keyboard","laptop","mouse","person","potted plant","scissors","tv","vase"],"natural_history_museum":["backpack","bench","book","chair","clock","person","potted plant","vase"],"nursery":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","teddy bear","tv","vase"],"nursing_home":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"oast_house":["bench","book","clock","person","potted plant","vase"],"ocean":["bird","boat","person","surfboard","umbrella"],"office":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"office_building":["backpack","bench","bicycle","book","bottle","bus","car","cell phone","chair","clock","handbag","keyboard","laptop","motorcycle","mouse","person","scissors","traffic light","truck","tv","umbrella"],"office_cubicles":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"oilrig":["backpack","car","person","truck"],"operating_room":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"orchard":["backpack","bird","cat","dog","person"],"orchestra_pit":["backpack","book","chair","clock","person","potted plant","vase"],"pagoda":["bench","book","clock","person","potted plant","vase"],"palace":["bench","book","clock","person","potted plant","vase"],"pantry":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"park":["backpack","bench","bicycle","bird","cat","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","sports ball","traffic light"],"parking_garage_indoor":["bicycle","book","car","cell phone","chair","clock","keyboard","laptop","motorcycle","mouse","parking meter","person","scissors","stop sign","traffic light","truck","tv"],"parking_garage_outdoor":["airplane","backpack","bench","bicycle","boat","bottle","bus","car","cell phone","handbag","motorcycle","parking meter","person","stop sign","traffic light","train","truck","umbrella"],"parking_lot":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"pasture":["backpack","bird","cat","cow","dog","horse","person","sheep"],"patio":["bench","bicycle","bird","bottle","cat","chair","cow","dog","horse","person","potted plant","sheep","umbrella"],"pavilion":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"pet_shop":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","umbrella","wine glass"],"pharmacy":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"phone_booth":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"physics_laboratory":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"picnic_area":["backpack","bench","bicycle","bird","cat","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","sports ball","traffic light"],"pier":["airplane","bench","bicycle","bird","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"pizzeria":["bottle","bowl","chair","cup","dining table","fork","knife","person","pizza","spoon","wine glass"],"playground":["backpack","bench","bicycle","bird","cat","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","sports ball","traffic light"],"playroom":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","teddy bear","tv","vase"],"plaza":["backpack","bench","bicycle","book","bottle","bus","car","cell phone","clock","handbag","motorcycle","person","potted plant","traffic light","truck","umbrella","vase"],"pond":["boat","person","surfboard","umbrella"],"porch":["backpack","bench","bicycle","bird","bottle","bus","car","cell phone","dog","frisbee","handbag","kite","motorcycle","person","skateboard","sports ball","traffic light","truck","umbrella"],"promenade":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"pub_indoor":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","tv","wine glass"],"racecourse":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"raceway":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"raft":["bench","boat","clock","fire hydrant","person","potted plant","surfboard","traffic light","umbrella"],"railroad_track":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"rainforest":["backpack","bird","cat","dog","person"],"reception":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"recreation_room":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"repair_shop":["backpack","book","bottle","bowl","cell phone","chair","clock","cup","dining table","fork","handbag","keyboard","knife","laptop","mouse","person","scissors","spoon","tv","umbrella","wine glass"],"residential_neighborhood":["backpack","bench","bicycle","bird","bottle","bus","car","cat","cell phone","cow","dog","handbag","horse","motorcycle","person","potted plant","sheep","traffic light","truck","umbrella"],"restaurant":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","wine glass"],"restaurant_kitchen":["bed","book","bottle","bowl","chair","clock","couch","cup","dining table","fork","knife","microwave","oven","person","potted plant","refrigerator","remote","sink","spoon","toaster","tv","vase","wine glass"],"restaurant_patio":["backpack","bench","bicycle","bottle","bowl","bus","car","cat","cell phone","chair","cup","dining table","dog","fork","handbag","knife","motorcycle","person","potted plant","spoon","traffic light","truck","umbrella","wine glass"],"rice_paddy":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light"],"river":["bird","boat","person","surfboard","umbrella"],"rock_arch":["backpack","person"],"roof_garden":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep","vase"],"rope_bridge":["bench","bicycle","bird","boat","bus","car","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","sports ball","traffic light","truck"],"ruin":["bench","book","clock","fire hydrant","person","potted plant","traffic light","vase"],"runway":["airplane","bicycle","boat","bus","car","motorcycle","parking meter","person","stop sign","traffic light","train","truck"],"sandbox":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"sauna":["baseball bat","baseball glove","bench","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket"],"schoolhouse":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"science_museum":["backpack","bench","book","chair","clock","person","potted plant","vase"],"server_room":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"shed":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep"],"shoe_shop":["backpack","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","spoon","umbrella","wine glass"],"shopfront":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"shopping_mall_indoor":["backpack","bench","bottle","bowl","cell phone","chair","cup","dining table","fork","handbag","knife","person","potted plant","spoon","umbrella","wine glass"],"shower":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"ski_resort":["backpack","bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep","skis","snowboard"],"ski_slope":["backpack","bench","bicycle","bird","boat","clock","dog","fire hydrant","frisbee","kite","person","potted plant","skateboard","skis","snowboard","sports ball","surfboard","traffic light","umbrella"],"sky":["backpack","person"],"skyscraper":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"slum":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"snowfield":["boat","person","surfboard","umbrella"],"soccer_field":["backpack","bench","bicycle","bird","cow","dog","frisbee","horse","kite","person","sheep","skateboard","sports ball"],"stable":["bed","book","bottle","cell phone","chair","clock","couch","cup","dog","horse","keyboard","laptop","mouse","person","potted plant","remote","scissors","tv","vase"],"stadium_baseball":["backpack","baseball bat","baseball glove","bench","bicycle","bird","bottle","dog","frisbee","kite","person","skateboard","sports ball"],"stadium_football":["backpack","bench","bicycle","bird","bottle","dog","frisbee","kite","person","skateboard","sports ball"],"stadium_soccer":["backpack","bench","bicycle","bird","bottle","dog","frisbee","kite","person","skateboard","sports ball"],"stage_indoor":["backpack","book","chair","clock","person","potted plant","vase"],"stage_outdoor":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"staircase":["backpack","bed","book","bottle","cell phone","chair","clock","couch","cup","keyboard","laptop","mouse","person","potted plant","remote","scissors","tv","vase"],"storage_room":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"street":["backpack","bench","bicycle","bottle","bus","car","cell phone","dog","fire hydrant","handbag","motorcycle","person","stop sign","traffic light","truck","umbrella"],"subway_station_platform":["backpack","bench","cell phone","handbag","person","suitcase","train","umbrella"],"supermarket":["apple","banana","bottle","bowl","broccoli","cake","carrot","chair","cup","dining table","fork","knife","orange","person","spoon","wine glass"],"sushi_bar":["bottle","bowl","chair","cup","dining table","fork","knife","person","spoon","tv","wine glass"],"swamp":["backpack","bird","boat","cat","dog","person","surfboard","umbrella"],"swimming_hole":["bench","boat","person","surfboard","umbrella"],"swimming_pool_indoor":["baseball bat","baseball glove","bench","chair","frisbee","kite","person","skateboard","skis","snowboard","sports ball","surfboard","tennis racket","umbrella"],"swimming_pool_outdoor":["bench","bicycle","bird","chair","dog","frisbee","kite","person","skateboard","sports ball","umbrella"],"synagogue_outdoor":["bench","book","clock","person","potted plant","vase"],"television_room":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"television_studio":["backpack","book","cell phone","chair","clock","keyboard","laptop","mouse","person","potted plant","scissors","tv","vase"],"temple_asia":["bench","book","clock","person","potted plant","vase"],"throne_room":["backpack","book","chair","clock","person","potted plant","vase"],"ticket_booth":["backpack","bench","bicycle","book","bottle","bowl","bus","car","cell phone","chair","clock","cup","dining table","fork","handbag","keyboard","knife","laptop","motorcycle","mouse","person","potted plant","scissors","spoon","traffic light","truck","tv","umbrella","vase","wine glass"],"topiary_garden":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light","vase"],"tower":["bench","book","clock","person","potted plant","vase"],"toyshop":["bicycle","book","bottle","bowl","chair","cup","dining table","fork","knife","person","skateboard","spoon","teddy bear","wine glass"],"train_interior":["backpack","bench","cell phone","handbag","person","suitcase","train","umbrella"],"train_station_platform":["backpack","bench","cell phone","handbag","person","suitcase","train","umbrella"],"tree_farm":["backpack","bird","cat","cow","dog","horse","person","sheep","truck"],"tree_house":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","frisbee","horse","kite","person","potted plant","sheep","skateboard","sports ball","traffic light"],"trench":["backpack","bench","car","clock","fire hydrant","person","potted plant","traffic light","truck"],"tundra":["backpack","person"],"underwater_ocean_deep":["bird","boat","person","surfboard","umbrella"],"utility_room":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"valley":["backpack","person"],"vegetable_garden":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light","vase"],"veterinarians_office":["book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"viaduct":["bench","book","clock","fire hydrant","person","potted plant","traffic light","vase"],"village":["backpack","bench","bicycle","bottle","bus","car","cell phone","handbag","motorcycle","person","traffic light","truck","umbrella"],"vineyard":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light"],"volcano":["backpack","person"],"volleyball_court_outdoor":["bench","bicycle","bird","dog","frisbee","kite","person","skateboard","sports ball"],"waiting_room":["backpack","bench","book","cell phone","chair","clock","keyboard","laptop","mouse","person","scissors","tv"],"water_park":["bench","bicycle","bird","dog","frisbee","kite","person","potted plant","skateboard","sports ball"],"water_tower":["backpack","car","person","truck"],"waterfall":["boat","person","surfboard","umbrella"],"watering_hole":["boat","person","surfboard","umbrella"],"wave":["boat","person","surfboard","umbrella"],"wet_bar":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase","wine glass"],"wheat_field":["backpack","bird","cat","cow","dog","horse","person","sheep"],"wind_farm":["backpack","bench","bird","car","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light","truck"],"windmill":["backpack","bench","bicycle","book","bottle","bus","car","cell phone","clock","handbag","motorcycle","person","potted plant","traffic light","truck","umbrella","vase"],"yard":["backpack","bench","bicycle","bird","cat","clock","cow","dog","fire hydrant","horse","person","potted plant","sheep","traffic light"],"youth_hostel":["bed","book","bottle","chair","clock","couch","cup","person","potted plant","remote","tv","vase"],"zen_garden":["bench","bicycle","bird","cat","cow","dog","horse","person","potted plant","sheep","vase"]}'

ALL_SCENES = json.loads(_SCENES_JSON)

SCENES = {
    name: classes
    for name, classes in ALL_SCENES.items()
    if SCENE_FILTER is None or name in SCENE_FILTER
}

lens = [len(v) for v in SCENES.values()]
print(f'Scenes      : {len(SCENES)}')
print(f'Avg classes : {statistics.mean(lens):.1f}')
print(f'Min classes : {min(lens)}')
print(f'Max classes : {max(lens)}')
print()
print('Sample:')
for name, cls in list(SCENES.items())[:6]:
    print(f'  {name:<35s} ({len(cls):2d}) {cls[:4]}')

## 4 — Generate Frozen Per-Scene Models

For each scene:
1. Load the base model
2. `model.set_classes(scene_classes)` — bakes CLIP text embeddings into weights
3. `model.save(scene_name.pt)` — serialises the frozen vocabulary

The saved `.pt` files do **not** require CLIP or `set_classes()` at load time.

In [ ]:
import time
from ultralytics import YOLO
from tqdm.auto import tqdm

print(f'Loading base model: {MODEL_NAME} ...')
t0 = time.perf_counter()
# Load once to trigger download, then reload per-scene for isolation
_ = YOLO(MODEL_NAME)
print(f'Base model ready in {(time.perf_counter()-t0)*1000:.0f} ms')

log = []   # (scene_name, num_classes, set_classes_ms, saved_path)
skipped = 0

for scene_name, classes in tqdm(SCENES.items(), desc='Generating'):
    out_path = OUT_DIR / f'{scene_name}.pt'

    if SKIP_EXISTING and out_path.exists():
        skipped += 1
        continue

    # Reload weights cleanly for each scene so embeddings are isolated
    model = YOLO(MODEL_NAME)

    t0 = time.perf_counter()
    model.set_classes(classes)
    t_sc = (time.perf_counter() - t0) * 1000

    model.save(str(out_path))
    log.append((scene_name, len(classes), t_sc, str(out_path)))

print(f'\nGenerated : {len(log)} models')
print(f'Skipped   : {skipped} (already exist)')
if log:
    avg_sc = sum(r[2] for r in log) / len(log)
    print(f'Avg set_classes(): {avg_sc:.0f} ms')

total_mb = sum(p.stat().st_size for p in OUT_DIR.glob('*.pt')) / 1e6
print(f'Total disk: {total_mb:.1f} MB  ({len(list(OUT_DIR.glob("*.pt")))} files)')

## 5 — Verify: Load a Frozen Model

In [ ]:
import numpy as np
from ultralytics import YOLO

test_scene = 'living_room'
frozen_path = OUT_DIR / f'{test_scene}.pt'

if not frozen_path.exists():
    # Fall back to any generated model
    pts = sorted(OUT_DIR.glob('*.pt'))
    if pts:
        frozen_path = pts[0]
        test_scene  = frozen_path.stem
    else:
        print('No frozen models found — run §4 first.')
        frozen_path = None

if frozen_path:
    frozen = YOLO(str(frozen_path))
    names  = list(frozen.names.values()) if isinstance(frozen.names, dict) else frozen.names
    print(f'Scene     : {test_scene}')
    print(f'Classes   : {names}')
    print(f'Num classes: {len(names)}')

    dummy  = np.zeros((640, 640, 3), dtype=np.uint8)
    result = frozen.predict(dummy, verbose=False, conf=0.25)
    print(f'Inference : OK ({len(result[0].boxes)} detections on blank image)')

## 6 — Benchmark: Dynamic `set_classes()` vs Frozen Model Load

Measures per-switch latency for both approaches over a random subset of scenes.

In [ ]:
import time, random, statistics
import numpy as np
from ultralytics import YOLO

BENCH_SCENES = random.sample(list(SCENES.keys()), min(20, len(SCENES)))
DUMMY        = np.zeros((640, 640, 3), dtype=np.uint8)
WARM         = 2
REPS         = 5

# ── Mode A: dynamic set_classes() ────────────────────────────────────────────
print('Mode A — dynamic set_classes() on shared model...')
dyn_model = YOLO(MODEL_NAME)
dyn_sc_times    = []
dyn_infer_times = []

for scene in BENCH_SCENES:
    cls = SCENES[scene]
    for i in range(WARM + REPS):
        t0 = time.perf_counter(); dyn_model.set_classes(cls); t1 = time.perf_counter()
        dyn_model.predict(DUMMY, verbose=False); t2 = time.perf_counter()
        if i >= WARM:
            dyn_sc_times.append((t1-t0)*1000)
            dyn_infer_times.append((t2-t1)*1000)

print(f'  set_classes(): {statistics.mean(dyn_sc_times):.1f} ± {statistics.stdev(dyn_sc_times):.1f} ms')
print(f'  inference    : {statistics.mean(dyn_infer_times):.1f} ± {statistics.stdev(dyn_infer_times):.1f} ms')

# ── Mode B: frozen model load ─────────────────────────────────────────────────
frz_scenes = [s for s in BENCH_SCENES if (OUT_DIR / f'{s}.pt').exists()]
frz_load_times  = []
frz_infer_times = []

if frz_scenes:
    print(f'\nMode B — frozen model load+infer ({len(frz_scenes)} scenes)...')
    for scene in frz_scenes:
        path = str(OUT_DIR / f'{scene}.pt')
        for i in range(WARM + REPS):
            t0 = time.perf_counter(); m = YOLO(path); t1 = time.perf_counter()
            m.predict(DUMMY, verbose=False); t2 = time.perf_counter()
            if i >= WARM:
                frz_load_times.append((t1-t0)*1000)
                frz_infer_times.append((t2-t1)*1000)
    print(f'  model load   : {statistics.mean(frz_load_times):.1f} ± {statistics.stdev(frz_load_times):.1f} ms')
    print(f'  inference    : {statistics.mean(frz_infer_times):.1f} ± {statistics.stdev(frz_infer_times):.1f} ms')
else:
    print('No frozen models found — run §4 first.')

In [ ]:
import matplotlib.pyplot as plt

if frz_load_times:
    labels = ['A: set_classes\n(switch only)', 'A: inference', 'B: model load', 'B: inference']
    means  = [statistics.mean(dyn_sc_times), statistics.mean(dyn_infer_times),
               statistics.mean(frz_load_times), statistics.mean(frz_infer_times)]
    stdevs = [statistics.stdev(dyn_sc_times), statistics.stdev(dyn_infer_times),
               statistics.stdev(frz_load_times), statistics.stdev(frz_infer_times)]
    colors = ['#3b82f6', '#93c5fd', '#f59e0b', '#fcd34d']

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(labels, means, yerr=stdevs, capsize=6, color=colors, edgecolor='none', width=0.5)
    ax.set_ylabel('Latency (ms)')
    ax.set_title('Dynamic set_classes() vs Frozen Per-Scene Model — COCO-80')
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{mean:.0f} ms', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('benchmark_coco80.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: benchmark_coco80.png')
else:
    print('Run Mode B benchmark first.')

## 7 — Summary Table

In [ ]:
from pathlib import Path

pt_files = sorted(OUT_DIR.glob('*.pt'))
total_mb = sum(p.stat().st_size for p in pt_files) / 1e6

print(f'Generated models : {len(pt_files)}')
print(f'Total disk       : {total_mb:.1f} MB')
print(f'Per-model avg    : {total_mb/max(len(pt_files),1):.1f} MB')
print()
print(f'{"Scene":<35s}  {"Classes":>7}  {"Size (KB)":>10}')
print('-' * 58)
for p in sorted(pt_files)[:20]:
    scene = p.stem
    n_cls = len(SCENES.get(scene, []))
    kb    = p.stat().st_size / 1024
    print(f'{scene:<35s}  {n_cls:>7d}  {kb:>8.0f} KB')
if len(pt_files) > 20:
    print(f'  ... and {len(pt_files)-20} more')

## 8 — Download Results (Colab)

In [ ]:
import shutil
from pathlib import Path

zip_name = 'scene_models_coco80'
zip_path = Path(f'{zip_name}.zip')
shutil.make_archive(zip_name, 'zip', OUT_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f'Packed: {zip_path}  ({size_mb:.1f} MB)')

try:
    from google.colab import files
    files.download(str(zip_path))
    # Also download benchmark chart if it exists
    bench = Path('benchmark_coco80.png')
    if bench.exists():
        files.download(str(bench))
except ImportError:
    print(f'Not in Colab — copy {zip_path} manually.')

---
## Appendix — Using the frozen models in the backend

1. Copy `scene_models_coco80/*.pt` → `backend/models/`

2. In the research page, switch Phase 2 to **Model Mode** (the toggle).

3. Update `model_file` per scene in `context.db` via `/manage-context`, or run:
```python
import sqlite3
conn = sqlite3.connect('backend/data/context.db')
cur  = conn.cursor()
# Example: set living_room to use its frozen model
cur.execute("UPDATE scene_context SET model_file='living_room.pt' WHERE scene_name='living_room'")
conn.commit(); conn.close()
```

4. Or update all scenes at once:
```python
import sqlite3, os
conn = sqlite3.connect('backend/data/context.db')
cur  = conn.cursor()
models_dir = 'backend/models'
cur.execute('SELECT scene_name FROM scene_context')
for (scene,) in cur.fetchall():
    if os.path.exists(os.path.join(models_dir, f'{scene}.pt')):
        cur.execute("UPDATE scene_context SET model_file=? WHERE scene_name=?",
                    (f'{scene}.pt', scene))
conn.commit(); conn.close()
print('Done')
```

### Latency comparison (Pi 5, CPU-only)
| Mode | Typical latency |
|---|---|
| Dynamic `set_classes()` on Pi 5 | **3–8 seconds** |
| Frozen model load on Pi 5 | **~300–600 ms** |

Frozen models are ~10× faster to switch on edge hardware.